# Funciones de Ventana e Índices

### Window Functions — OVER()

In [148]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE employees (
    id INTEGER PRIMARY KEY, name TEXT, department TEXT, salary REAL)''')
cursor.executemany('INSERT INTO employees VALUES (?,?,?,?)', [
    (1, 'Alice', 'Engineering', 75000), (2, 'Bob', 'Marketing', 55000),
    (3, 'Carol', 'Engineering', 82000), (4, 'David', 'HR', 48000),
    (5, 'Eva', 'Marketing', 61000), (6, 'Frank', 'Engineering', 79000),
    (7, 'Grace', 'HR', 52000), (8, 'Henry', 'Marketing', 58000),
])
conn.commit()

# ROW_NUMBER, RANK, DENSE_RANK
cursor.execute("""
    SELECT
        id, name, department, salary,
        ROW_NUMBER() OVER (ORDER BY salary DESC)                        AS row_num,
        RANK()       OVER (ORDER BY salary DESC)                        AS rank_global,
        ROW_NUMBER() OVER (PARTITION BY department ORDER BY salary DESC) AS rank_in_dept
    FROM employees
    ORDER BY department desc
""")
print(f"{'Nombre':<10} {'Depto':<15} {'Salario':<15} {'Row#':>5} {'Rank':>5} {'DeptRk':>7}")
print("-" * 55)
for row in cursor.fetchall():
    # print(row)
    print(f"{row[0]:<4} | {row[1]:<10} | {row[2]:<12} | ${row[3]:<8} | {row[4]:<2} | {row[5]:<2} | {row[6]:<2} |")

Nombre     Depto           Salario          Row#  Rank  DeptRk
-------------------------------------------------------
5    | Eva        | Marketing    | $61000.0  | 4  | 4  | 1  |
8    | Henry      | Marketing    | $58000.0  | 5  | 5  | 2  |
2    | Bob        | Marketing    | $55000.0  | 6  | 6  | 3  |
7    | Grace      | HR           | $52000.0  | 7  | 7  | 1  |
4    | David      | HR           | $48000.0  | 8  | 8  | 2  |
3    | Carol      | Engineering  | $82000.0  | 1  | 1  | 1  |
6    | Frank      | Engineering  | $79000.0  | 2  | 2  | 2  |
1    | Alice      | Engineering  | $75000.0  | 3  | 3  | 3  |


### PARTITION BY — Ventanas por Grupo

In [149]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE sales (
    id INTEGER PRIMARY KEY, rep TEXT, region TEXT, amount REAL, sale_date TEXT)''')
cursor.executemany('INSERT INTO sales VALUES (?,?,?,?,?)', [
    (1, 'Alice', 'Norte', 5000, '2024-01-10'), (2, 'Bob', 'Sur', 3500, '2024-01-12'),
    (3, 'Carol', 'Norte', 7200, '2024-01-15'), (4,
                                                'David', 'Sur', 2800, '2024-01-18'),
    (5, 'Alice', 'Norte', 6100, '2024-02-05'), (6, 'Bob', 'Sur', 4200, '2024-02-08'),
    (7, 'Carol', 'Norte', 8900, '2024-02-12'), (8,
                                                'David', 'Sur', 3600, '2024-02-15'),
    (9, 'Eva',  'Este', 5500, '2024-02-20'), (10, 'Eva', 'Este', 4800, '2024-03-01'),
])
conn.commit()

# Porcentaje de cada venta dentro de su región
cursor.execute("""
    SELECT
        rep, region, amount,
        SUM(amount)   OVER (PARTITION BY region)                            AS region_total,
        ROUND(amount * 100.0 / SUM(amount) OVER (PARTITION BY region), 1)   AS pct_region,
        AVG(amount)   OVER (PARTITION BY region)                            AS region_avg,
        RANK()        OVER (PARTITION BY region ORDER BY amount DESC)       AS rank_in_region
    FROM sales
    ORDER BY region, amount DESC
""")
print(f"{'Rep':<8} {'Región':<6} {'Venta':>7} {'Total Reg':>10} {'%Reg':>6} {'Rank':>5}")
print("-" * 50)
for row in cursor.fetchall():
    print(
        f"| {row[0]:<8} | {row[1]:<6} | ${row[2]:>5,} | ${row[3]:>8,} | {row[4]:>5}% | {row[6]:>5}")

Rep      Región   Venta  Total Reg   %Reg  Rank
--------------------------------------------------
| Eva      | Este   | $5,500.0 | $10,300.0 |  53.4% |     1
| Eva      | Este   | $4,800.0 | $10,300.0 |  46.6% |     2
| Carol    | Norte  | $8,900.0 | $27,200.0 |  32.7% |     1
| Carol    | Norte  | $7,200.0 | $27,200.0 |  26.5% |     2
| Alice    | Norte  | $6,100.0 | $27,200.0 |  22.4% |     3
| Alice    | Norte  | $5,000.0 | $27,200.0 |  18.4% |     4
| Bob      | Sur    | $4,200.0 | $14,100.0 |  29.8% |     1
| David    | Sur    | $3,600.0 | $14,100.0 |  25.5% |     2
| Bob      | Sur    | $3,500.0 | $14,100.0 |  24.8% |     3
| David    | Sur    | $2,800.0 | $14,100.0 |  19.9% |     4


### LAG y LEAD — Comparar con Filas Anteriores/Siguientes

In [154]:
import sqlite3

conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

cursor.execute('''CREATE TABLE monthly_revenue (
    id INTEGER PRIMARY KEY, month TEXT, revenue REAL, region TEXT)''')
cursor.executemany('INSERT INTO monthly_revenue VALUES (?,?,?,?)', [
    (1,'2024-01',45000,'Norte'),(2,'2024-02',52000,'Norte'),(3,'2024-03',48000,'Norte'),
    (4,'2024-04',61000,'Norte'),(5,'2024-05',57000,'Norte'),(6,'2024-06',69000,'Norte'),
    (7,'2024-01',32000,'Sur'),(8,'2024-02',29000,'Sur'),(9,'2024-03',35000,'Sur'),
    (10,'2024-04',38000,'Sur'),(11,'2024-05',41000,'Sur'),(12,'2024-06',44000,'Sur'),
])
conn.commit()

# Crecimiento MoM (mes a mes) con LAG
cursor.execute("""
    SELECT
        month, region, revenue,
        LAG(revenue)  OVER (PARTITION BY region ORDER BY month) AS prev_month,
        revenue - LAG(revenue, 1, revenue)
                      OVER (PARTITION BY region ORDER BY month) AS mom_change,
        ROUND((revenue - LAG(revenue, 1, revenue)
                         OVER (PARTITION BY region ORDER BY month))
              * 100.0 / LAG(revenue, 1, revenue)
                         OVER (PARTITION BY region ORDER BY month), 1) AS mom_pct
    FROM monthly_revenue
    ORDER BY region, month
""")
print(f"{'Mes':<10} {'Region':<10} {'Revenue':>9} {'Anterior':>12} {'Cambio':>12} {'%':>10}")
print("-" * 80)
for row in cursor.fetchall():
    prev = f"${row[3]:>10,}" if row[3] else "  N/A"
    change = f"+${row[4]:>10,}" if row[4] and row[4] > 0 else f"${row[4]:>10,}" if row[4] else "  N/A"
    pct = f"${row[5]:>8}%" if row[5] else "  N/A"
    print(f"{row[0]:<10} {row[1]:<10} ${row[2]:>9,} {prev:>12} {change:>12} {pct:>10}")


Mes        Region       Revenue     Anterior       Cambio          %
--------------------------------------------------------------------------------
2024-01    Norte      $ 45,000.0          N/A          N/A        N/A
2024-02    Norte      $ 52,000.0  $  45,000.0 +$   7,000.0 $    15.6%
2024-03    Norte      $ 48,000.0  $  52,000.0  $  -4,000.0 $    -7.7%
2024-04    Norte      $ 61,000.0  $  48,000.0 +$  13,000.0 $    27.1%
2024-05    Norte      $ 57,000.0  $  61,000.0  $  -4,000.0 $    -6.6%
2024-06    Norte      $ 69,000.0  $  57,000.0 +$  12,000.0 $    21.1%
2024-01    Sur        $ 32,000.0          N/A          N/A        N/A
2024-02    Sur        $ 29,000.0  $  32,000.0  $  -3,000.0 $    -9.4%
2024-03    Sur        $ 35,000.0  $  29,000.0 +$   6,000.0 $    20.7%
2024-04    Sur        $ 38,000.0  $  35,000.0 +$   3,000.0 $     8.6%
2024-05    Sur        $ 41,000.0  $  38,000.0 +$   3,000.0 $     7.9%
2024-06    Sur        $ 44,000.0  $  41,000.0 +$   3,000.0 $     7.3%


## Mine

In [151]:
import psycopg2

conn = psycopg2.connect(
    host='localhost', port=5432,
    dbname='movies', user='postgres', password='postgres'
)
cursor = conn.cursor()

cursor.execute(
    """
    Select 
        Concat(c.first_name, ' ', c.last_name) as name,
        DATE(r.rental_date) as rental_date,
        r.rental_date::time as rental_time,
        p.amount as amount,
        SUM(amount) OVER(PARTITION BY DATE(rental_date)) as total,
        ((100 / (SUM(amount) OVER(PARTITION BY DATE(rental_date)))) * p.amount) as percentage_amount_by_reg,
        AVG(amount) OVER(PARTITION BY DATE(rental_date)) as avg_by_date
    from customer AS c
    LEFT JOIN rental r ON c.customer_id = r.customer_id
    LEFT JOIN payment p on r.rental_id = p.rental_id
    ORDER BY r.rental_date
    """
)

print(f"| {'Nombre':<20} | {'Fecha':<12} | {'Hora':<10} | {'Monto':>8} | {'Total':>10} | {'%':>10} | {'Prom':>8} |")
print("-" * 95)

for r in cursor.fetchall():
    name, rental_date, rental_time, amount, total, pct, avg = r
    pct_str = f"{pct:.2f}%" if pct else "N/A"
    print(f"| {name:<20} | {rental_date} | {rental_time} | ${amount:>6} | ${total:>8} | {pct_str:>10} | ${avg:>6} |")


| Nombre               | Fecha        | Hora       |    Monto |      Total |          % |     Prom |
-----------------------------------------------------------------------------------------------
| CHARLOTTE HUNTER     | 2005-05-24 | 19:53:30 | $  2.99 | $  109.76 |      2.72% | $4.5733333333333333 |
| TOMMY COLLAZO        | 2005-05-24 | 19:54:33 | $  2.99 | $  109.76 |      2.72% | $4.5733333333333333 |
| MANUEL MURRELL       | 2005-05-24 | 20:03:39 | $  3.99 | $  109.76 |      3.64% | $4.5733333333333333 |
| ANDREW PURDY         | 2005-05-24 | 20:04:41 | $  4.99 | $  109.76 |      4.55% | $4.5733333333333333 |
| DELORES HANSEN       | 2005-05-24 | 20:05:21 | $  6.99 | $  109.76 |      6.37% | $4.5733333333333333 |
| NELSON CHRISTENSON   | 2005-05-24 | 20:08:07 | $  0.99 | $  109.76 |      0.90% | $4.5733333333333333 |
| CASSANDRA WALTERS    | 2005-05-24 | 20:11:53 | $  1.99 | $  109.76 |      1.81% | $4.5733333333333333 |
| MINNIE ROMERO        | 2005-05-24 | 20:31:46 | $  4.99 | $ 